# Practice 1 · From Text to Token Embeddings

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

Text becomes model input in two stages: a tokenizer maps text to token IDs, and an embedding
matrix maps those IDs to vectors. This practice follows that path and then uses cosine
similarity to compare word representations.

### II. Task Requirements

1. **Q1** — complete *TinyTokenizer.encode(text)* using the tokenizer vocabulary.
2. **Q2** — write *lookup_embeddings(ids, embedding_matrix)*.
3. **Q3** — write *cosine_similarity(a, b)* and compare word vectors.

Steps: open the notebook in Haima, write the 🟢 parts, and run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *encode(text)* | token IDs for the input text | a list of Python integers |
| *lookup_embeddings(...)* | vectors for the token IDs | shape *(number of tokens, embedding dimension)* |
| *cosine_similarity(a, b)* | similarity of two vectors | a float in *[-1, 1]* |
| zero vector | undefined cosine similarity | raise *ValueError* |

### IV. Who does what

- **Q1**: required for undergraduate students; postgraduates do it as well.
- **Q2–Q3**: required for postgraduate students; optional for undergraduates.
  An undergraduate who finishes advanced questions gets bonus marks; leaving them unfinished costs nothing.


## Question 1 · From text to token IDs

Run the provided toy tokenizer on Chinese, English, and code. It splits English into words,
Chinese into individual characters, and code into identifiers, numbers, and symbols. This is
an illustrative tokenizer; real language models commonly use learned subword tokenizers.

Complete *TinyTokenizer.encode(text)*: call *self.tokenize(text)*, then map each token to its
ID using *self.vocab*. Use *self.unk_id* for a token that is absent from the vocabulary.
Return a list of Python integers.


In [1]:
# 🟢 EDIT: set SID to the last two digits of your student ID.
SID = 23

import math
import re
import torch
import torch.nn.functional as F

torch.manual_seed(0)


# 🟡 READ: a small local tokenizer; no download or network access is needed.
class TinyTokenizer:
    def __init__(self, corpus):
        tokens = []
        for text in corpus:
            tokens.extend(self.tokenize(text))
        vocab = sorted(set(tokens))
        self.vocab = {token: i + 1 for i, token in enumerate(vocab)}
        self.unk_id = 0

    def tokenize(self, text):
        return re.findall(r"[\u4e00-\u9fff]|[A-Za-z]+|\d+|[^\w\s]", text, flags=re.UNICODE)

    def encode(self, text):
        """Return token IDs, using unk_id for an unknown token."""
        # 🟢 CODE HERE: tokenize the text and look up every token in self.vocab.
        tokens = self.tokenize(text)
        return [self.vocab.get(token,self.unk_id) for token in tokens]


texts = [
    "I like this course.",
    "我喜欢这门课。",
    "x = 42; print(x)",
]
tokenizer = TinyTokenizer(texts)
for text in texts:
    print(text)
    print("tokens:", tokenizer.tokenize(text))
    print("ids:   ", tokenizer.encode(text))


I like this course.
tokens: ['I', 'like', 'this', 'course', '.']
ids:    [7, 9, 11, 8, 3]
我喜欢这门课。
tokens: ['我', '喜', '欢', '这', '门', '课', '。']
ids:    [15, 14, 16, 18, 19, 17, 13]
x = 42; print(x)
tokens: ['x', '=', '42', ';', 'print', '(', 'x', ')']
ids:    [12, 6, 4, 5, 10, 1, 12, 2]


## Question 2 · Look up token embeddings

Write *lookup_embeddings(ids, embedding_matrix)*. The matrix has one row per vocabulary ID,
including the unknown-token row. Return the rows corresponding to *ids* in their original order.

Convert *ids* to a 1-D LongTensor and use *embedding_matrix.index_select(0, ids_tensor)*.
For *n* tokens and an embedding dimension of *d*, the output shape must be *(n, d)*.
Do not train or modify the embedding matrix.


In [2]:
# 🟢 EDIT: convert the IDs and select their rows from the embedding matrix.
def lookup_embeddings(ids, embedding_matrix):
    """Return one embedding row per token ID, preserving order."""
    ids_tensor = torch.tensor(ids, dtype=torch.long)
    return embedding_matrix.index_select(0, ids_tensor)


# ⚪ RUN: use a fixed local embedding matrix; no training is required.
embedding_matrix = torch.randn(len(tokenizer.vocab) + 1, 4)
ids = tokenizer.encode(texts[0])
vectors = lookup_embeddings(ids, embedding_matrix)
print("token IDs:", ids)
print("embedding shape:", tuple(vectors.shape))
print("first token vector:", vectors[0])


token IDs: [7, 9, 11, 8, 3]
embedding shape: (5, 4)
first token vector: tensor([ 1.3894,  1.5863,  0.9463, -0.8437])


## Question 3 · Compare word vectors with cosine similarity

Write *cosine_similarity(a, b)* for two nonzero 1-D vectors with the same shape:

$$
\cos(a, b)=\frac{a\cdot b}{\lVert a\rVert_2\,\lVert b\rVert_2}
$$

Return the result as a Python float. Raise *ValueError* if the shapes differ, either input is
not 1-D, or either vector has zero norm. You may use *F.cosine_similarity(a, b, dim=0)* after
checking these conditions.


In [4]:
# 🟢 EDIT: check the inputs and compute cosine similarity.
def cosine_similarity(a, b):
    """Return cosine similarity between two nonzero 1-D vectors."""
    if a.ndim != 1 or b.ndim != 1:
        raise ValueError("a and b must be 1-D vectors")

    if a.shape != b.shape:
        raise ValueError("a and b must have the same shape")

    if torch.linalg.vector_norm(a).item() == 0 or torch.linalg.vector_norm(b).item() == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")

    return F.cosine_similarity(a, b, dim=0).item()


# ⚪ RUN: compare a related pair with an unrelated pair.
word_vectors = {
    "motel": torch.tensor([1.0, 0.0, 0.0]),
    "hotel": torch.tensor([0.95, 0.05, 0.0]),
    "banana": torch.tensor([0.0, 1.0, 0.0]),
}
related = cosine_similarity(word_vectors["motel"], word_vectors["hotel"])
unrelated = cosine_similarity(word_vectors["motel"], word_vectors["banana"])
print(f"cosine(motel, hotel) = {related:.4f}")
print(f"cosine(motel, banana) = {unrelated:.4f}")


cosine(motel, hotel) = 0.9986
cosine(motel, banana) = 0.0000
